<a href="https://colab.research.google.com/github/anushtuprouth04-cmd/Fraud-detection/blob/main/household_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PROJECT: PREDICTING APPLIANCE ENERGY CONSUMPTION
# ============================================================

# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# ============================================================
# LOAD DATASET
# ============================================================

# Upload CSV into Google Colab

import os
print("Current working directory:", os.getcwd())
print("Files in current directory:", os.listdir())

df = pd.read_csv("energydata_complete.csv")

print("Dataset Loaded Successfully")
print("\nShape of Dataset:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

# ============================================================
# BASIC INFORMATION
# ============================================================

print("\nDataset Information")
print(df.info())

print("\nStatistical Summary")
display(df.describe())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Records")
print(df.duplicated().sum())

# ============================================================
# CONVERT DATE COLUMN
# ============================================================

df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y %H:%M")

# Extract useful features

df["Hour"] = df["date"].dt.hour

df["Day"] = df["date"].dt.day

df["Month"] = df["date"].dt.month

df["Weekday"] = df["date"].dt.dayofweek

df["Weekend"] = np.where(df["Weekday"] >= 5, 1, 0)

print("Date Features Created Successfully")

display(
    df[
        ["date","Hour","Day","Month","Weekday","Weekend"]
    ].head()
)

# ============================================================
# MISSING VALUE HEATMAP
# ============================================================

plt.figure(figsize=(14,5))
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

# ============================================================
# TARGET VARIABLE DISTRIBUTION
# ============================================================

fig = px.histogram(
    df,
    x="Appliances",
    nbins=60,
    title="Distribution of Appliance Energy Consumption"
)

fig.show()

# ============================================================
# ENERGY CONSUMPTION OVER TIME
# ============================================================

fig = px.line(
    df,
    x="date",
    y="Appliances",
    title="Energy Consumption Over Time"
)

fig.show()

# ============================================================
# CORRELATION MATRIX
# ============================================================

corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(20,14))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap")
plt.show()

# ============================================================
# FEATURES MOST RELATED TO TARGET
# ============================================================

target_corr = corr_matrix["Appliances"]

target_corr = target_corr.sort_values(
    ascending=False
)

print(target_corr.head(15))

# ============================================================
# FEATURE SELECTION
# ============================================================

# Date column no longer needed

df_model = df.drop("date", axis=1)

# Random variables add no useful information

df_model = df_model.drop(
    ["rv1","rv2"],
    axis=1
)

print(df_model.shape)


# ============================================================
# SPLIT FEATURES & TARGET
# ============================================================

X = df_model.drop(
    "Appliances",
    axis=1
)

y = df_model["Appliances"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples:", X_train.shape)
print("Testing Samples:", X_test.shape)

# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling Completed")


# ============================================================
# LINEAR REGRESSION
# ============================================================

lr = LinearRegression()

lr.fit(
    X_train_scaled,
    y_train
)

lr_pred = lr.predict(
    X_test_scaled
)

print("Linear Regression Training Completed")

# ============================================================
# RANDOM FOREST
# ============================================================

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict(
    X_test
)

print("Random Forest Training Completed")

# ============================================================
# RANDOM FOREST TUNING
# ============================================================

param_grid = {
    "n_estimators":[100,200],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=42
    ),
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

best_rf = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)
# ============================================================
# BEST RANDOM FOREST PREDICTION
# ============================================================

best_rf_pred = best_rf.predict(
    X_test
)
# ============================================================
# MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    actual,
    predicted,
    model_name
):

    mse = mean_squared_error(
        actual,
        predicted
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        actual,
        predicted
    )

    r2 = r2_score(
        actual,
        predicted
    )

    print(f"\n{model_name}")
    print("-"*40)

    print("MSE :", mse)
    print("RMSE:", rmse)
    print("MAE :", mae)
    print("R2  :", r2)

    return [model_name,mse,rmse,mae,r2]
    # ============================================================
# COMPARE MODELS
# ============================================================

results = []

results.append(
    evaluate_model(
        y_test,
        lr_pred,
        "Linear Regression"
    )
)

results.append(
    evaluate_model(
        y_test,
        best_rf_pred,
        "Random Forest"
    )
)

comparison = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MSE",
        "RMSE",
        "MAE",
        "R2"
    ]
)

display(comparison)
# ============================================================
# ACTUAL VS PREDICTED
# ============================================================

plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    best_rf_pred,
    alpha=0.5
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")

plt.title(
    "Actual vs Predicted Energy Consumption"
)

plt.show()
# ============================================================
# RESIDUAL ANALYSIS
# ============================================================

residuals = y_test - best_rf_pred

plt.figure(figsize=(8,6))

sns.histplot(
    residuals,
    kde=True
)

plt.title(
    "Residual Distribution"
)

plt.show()


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance":
        best_rf.feature_importances_
    }
)

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

display(
    importance.head(15)
)

plt.figure(figsize=(10,7))

sns.barplot(
    data=importance.head(15),
    x="Importance",
    y="Feature"
)

plt.title(
    "Top 15 Important Features"
)

plt.show()


# ============================================================
# SAVE MODEL
# ============================================================

import joblib

joblib.dump(
    best_rf,
    "Appliance_Energy_Model.pkl"
)

joblib.dump(
    scaler,
    "Scaler.pkl"
)

print("Model Saved Successfully")
# ============================================================
# FINAL CONCLUSION
# ============================================================

print("""
PROJECT SUMMARY
------------------------------------

1. Dataset cleaned and analyzed.

2. Date features engineered.

3. Correlation analysis performed.

4. Linear Regression developed.

5. Random Forest developed.

6. Hyperparameter tuning performed.

7. Models compared using:

   - MSE
   - RMSE
   - MAE
   - R² Score

8. Feature importance analyzed.

9. Best model saved for deployment.

BUSINESS IMPACT
------------------------------------

• Predict household energy consumption.

• Improve energy-saving recommendations.

• Reduce electricity bills.

• Improve smart-grid planning.

• Support sustainable energy management.
""")

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving descripton.txt to descripton (1).txt
Saving energydata_complete(1).xlsx to energydata_complete(1) (1).xlsx
Saving energydata_complete.csv to energydata_complete (1).csv
Saving energydata_complete.xlsx to energydata_complete (1).xlsx


After uploading, you can verify the file is present by listing the directory contents again if needed. Then, you can re-run the cell that reads the CSV file (`G8yw3VoZQuEa`).